In [4]:
# prepare_dataset
import json
import os
from pathlib import Path
import random

root_dir = "./compressed_dataset"  # Your folder
countries = {}
for country_dir in Path(root_dir).iterdir():
    if country_dir.is_dir():
        img_files = list(country_dir.glob("canvas_*.jpg"))  # Matches your structure
        if len(img_files) > 0:
            countries[country_dir.name] = [str(img) for img in img_files]  # Full paths

# Sample ~12k total (balanced: ~80 per country if possible)
samples = []
random.seed(42)
for country, imgs in countries.items():
    num_sample = min(80, len(imgs))  # Cap for balance
    sampled_imgs = random.sample(imgs, num_sample)
    samples.extend([{"image_path": img_path, "country": country} for img_path in sampled_imgs])

# Shuffle and save (for 10k generation)
random.shuffle(samples)
samples = samples[:12000]  # Safety buffer
with open("geoguessr_samples_12k.json", "w") as f:
    json.dump(samples, f, indent=2)

print(f"Prepared {len(samples)} samples across {len(countries)} countries.")
print("Example:", samples[0])

Prepared 6434 samples across 124 countries.
Example: {'image_path': 'compressed_dataset/Switzerland/canvas_1630050127.jpg', 'country': 'Switzerland'}


In [ ]:
# generate_refutation_cot.py ← ULTRA-FAST VERSION (ready to run)
import openai, json, base64, time, io, os
import random

client = openai.OpenAI(
    api_key="",  # ← put your SuperGrok key here
    base_url="https://api.x.ai/v1"
)

# Load samples and coords
with open("geoguessr_samples_12k.json", "r") as f:
    samples = json.load(f)
with open("folder_to_coords.json", "r") as f:
    coords = json.load(f)

results = []
random.seed(42)

for i, ex in enumerate(samples):
    if i >= 1000: break

    img_path = ex["image_path"]
    country = ex["country"]
    if country not in coords:
        print(f"Skipping {country}—no coords")
        continue

    lat = coords[country]["lat"] + random.uniform(-0.5, 0.5)
    lon = coords[country]["lon"] + random.uniform(-0.5, 0.5)
    lat_str = f"{round(abs(lat), 4)}°{'S' if lat < 0 else 'N'}"
    lon_str = f"{round(abs(lon), 4)}°{'W' if lon < 0 else 'E'}"

    from PIL import Image
    img = Image.open(img_path).convert("RGB")
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode()

    prompt = f"""<image>
You are the world's best GeoGuessr player. For this photo taken at {lat_str}, {lon_str} in {country}, write a detailed refutation-style reasoning in exactly this format (no markdown):
Image location: {country}
First I thought this might be [wrong place 1] because [reason].
But that cannot be, because [2–4 refuting clues].
Another possibility was [wrong place 2] due to [similarity].
However, [2–4 refuting clues] rule that out.
The correct clues are: [list 4–6 positive clues: signs, plates, vegetation, architecture, etc.].
This photo was taken in {country}.
Latitude: {lat_str}, Longitude: {lon_str}"""

    try:
        resp = client.chat.completions.create(
            model="grok-4-1-fast-reasoning",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            }],
            max_tokens=768,
            temperature=0.7,
            top_p=0.9
        )
        text = resp.choices[0].message.content
        results.append({
            "image": img_path,
            "messages": [{"role": "user", "content": "<image>\nThink step by step..."}, {"role": "assistant", "content": text}],
            "country": country,
            "true_lat": lat,
            "true_lon": lon
        })
        print(f"{i+1}/1000 Success {country} – {resp.usage.total_tokens} tokens")

    except Exception as e:
        print(f"Error {i}: {e}")
        time.sleep(3)
        continue

    # CHANGE #1: SuperGrok allows 480 RPM → you can go extremely fast
    if i % 500 == 0 and i > 0:
        with open(f"refutation_10k_checkpoint_{i}.json", "w") as f:
            json.dump(results, f, indent=2)

    # CHANGE #2: This is the only line you needed to change!
    time.sleep(0.001)   # ← 0.001 sec = ~60,000 requests/hour → 10× faster than 0.2

# Final save
with open("refutation_1k_final.json", "w") as f:
    json.dump(results, f, indent=2)

print("FINISHED! 1,000 perfect refutation CoT examples ready!")

1/1000 Success Switzerland – 2998 tokens
2/1000 Success Dominican Republic – 2097 tokens
3/1000 Success Albania – 1697 tokens
4/1000 Success Colombia – 2266 tokens
5/1000 Success Finland – 2002 tokens
6/1000 Success Cambodia – 1557 tokens
7/1000 Success Costa Rica – 2096 tokens
8/1000 Success Mongolia – 1703 tokens
9/1000 Success Nigeria – 2229 tokens
10/1000 Success Germany – 1853 tokens
11/1000 Success Albania – 2186 tokens
12/1000 Success Brazil – 1804 tokens
13/1000 Success Slovakia – 2374 tokens
14/1000 Success United States – 3148 tokens
15/1000 Success Portugal – 2165 tokens
16/1000 Success Poland – 2338 tokens
17/1000 Success Czechia – 1981 tokens
18/1000 Success Tunisia – 2880 tokens
19/1000 Success United Kingdom – 2287 tokens
20/1000 Success Tunisia – 1723 tokens
21/1000 Success Ecuador – 2020 tokens
22/1000 Success Chile – 2965 tokens
23/1000 Success United Arab Emirates – 2041 tokens
24/1000 Success Kyrgyzstan – 2147 tokens
25/1000 Success Latvia – 2251 tokens
26/1000 Succ

In [ ]:
# generate_normal_cot.py ← ULTRA-FAST VERSION (ready to run)
import openai, json, base64, time, io, os
import random

client = openai.OpenAI(
    api_key="",  # ← put your SuperGrok key here
    base_url="https://api.x.ai/v1"
)

# Load samples and coords
with open("geoguessr_samples_12k.json", "r") as f:
    samples = json.load(f)
with open("folder_to_coords.json", "r") as f:
    coords = json.load(f)

results = []
random.seed(42)

for i, ex in enumerate(samples):
    if i >= 1000: break

    img_path = ex["image_path"]
    country = ex["country"]
    if country not in coords:
        print(f"Skipping {country}—no coords")
        continue

    lat = coords[country]["lat"] + random.uniform(-0.5, 0.5)
    lon = coords[country]["lon"] + random.uniform(-0.5, 0.5)
    lat_str = f"{round(abs(lat), 4)}°{'S' if lat < 0 else 'N'}"
    lon_str = f"{round(abs(lon), 4)}°{'W' if lon < 0 else 'E'}"

    from PIL import Image
    img = Image.open(img_path).convert("RGB")
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode()

    prompt = f"""<image>
    You are the world's best GeoGuessr player.
    Examine this photo taken in {country} at {lat_str}, {lon_str}.
    Reason step by step about where in the world it was taken.
    List every visual clue you see (language, plates, vegetation, architecture, etc.)
    Then give your final guess.
    
    Final format:
    Latitude: {lat_str}, Longitude: {lon_str}"""

    try:
        resp = client.chat.completions.create(
            model="grok-4-1-fast-reasoning",
            messages=[{
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            }],
            max_tokens=768,
            temperature=0.7,
            top_p=0.9
        )
        text = resp.choices[0].message.content
        results.append({
            "image": img_path,
            "messages": [{"role": "user", "content": "<image>\nThink step by step..."}, {"role": "assistant", "content": text}],
            "country": country,
            "true_lat": lat,
            "true_lon": lon
        })
        print(f"{i+1}/1000 Success {country} – {resp.usage.total_tokens} tokens")

    except Exception as e:
        print(f"Error {i}: {e}")
        time.sleep(3)
        continue

    # CHANGE #1: SuperGrok allows 480 RPM → you can go extremely fast
    if i % 500 == 0 and i > 0:
        with open(f"refutation_10k_checkpoint_{i}.json", "w") as f:
            json.dump(results, f, indent=2)

    # CHANGE #2: This is the only line you needed to change!
    time.sleep(0.001)   # ← 0.001 sec = ~60,000 requests/hour → 10× faster than 0.2

# Final save
with open("norma_1k_final.json", "w") as f:
    json.dump(results, f, indent=2)

print("FINISHED! 1,000 perfect normal CoT examples ready!")

1/1000 Success Switzerland – 3843 tokens
2/1000 Success Dominican Republic – 1981 tokens
